In [1]:
# 06_aggregate_and_statistics.ipynb
# Merge the per-method raw CSVs and produce the revision's quantitative results:
#  R8-4  mean / std / 95% CI (Student-t) over seeds for every metric.
#  R8-1  separate Friedman + Nemenyi test PER classifier (RF, LGBM, MLP), on the
#        across-seed mean AUC (one value per dataset x method x classifier).
#  R8-2  Q-G-CTGAN (full) vs G-CTGAN (unfiltered), per classifier, all datasets:
#        mean AUC, delta, Wilcoxon signed-rank p, win/tie/loss.
# All tables are written to ../results/tables. ENV: pandas, numpy, scipy, scikit-posthocs.
import os, glob
import numpy as np
import pandas as pd
from scipy import stats
import scikit_posthocs as sp

TABLES_DIR = "../results/tables"
METRICS = ["AUC", "PR_AUC", "F1", "Precision", "Recall", "G_mean", "Balanced_Acc"]
CLASSIFIERS = ("RF", "LGBM", "MLP")
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN", "CTGAN_MOS",
                "ctdGAN", "CTAB-GAN+", "G-CTGAN", "Q-G-CTGAN"]
LABELS = {"None": "None", "SMOTE": "SMOTE", "ADASYN": "ADASYN", "G-SMOTE": "G-SMOTE",
          "KMeans_CTGAN": "K-means CTGAN", "CTGAN_MOS": "CTGAN-MOS", "ctdGAN": "ctdGAN",
          "CTAB-GAN+": "CTAB-GAN+", "G-CTGAN": "G-CTGAN (unfiltered)", "Q-G-CTGAN": "Q-G-CTGAN"}

# ---- merge every raw_*.csv in results/tables ----
raws = sorted(glob.glob(os.path.join(TABLES_DIR, "raw_*.csv")))
assert raws, "no raw_*.csv found in ../results/tables -- run notebooks 01-05 first"
raw = pd.concat([pd.read_csv(f, keep_default_na=False) for f in raws], ignore_index=True)
for c in METRICS + ["seed"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw.to_csv(os.path.join(TABLES_DIR, "05_multiseed_raw.csv"), index=False)
print("merged", len(raw), "rows from", [os.path.basename(f) for f in raws])

# ---- R8-4: mean / std / 95% CI per (dataset, method, classifier) ----
rows = []
for (ds, m, clf, mg), sub in raw.groupby(["dataset", "method", "classifier", "method_group"]):
    rec = {"dataset": ds, "method": m, "classifier": clf, "method_group": mg,
           "n_seeds": sub["seed"].nunique()}
    for met in METRICS:
        v = sub[met].dropna().values; n = len(v)
        mean = v.mean() if n else np.nan
        sd = v.std(ddof=1) if n > 1 else 0.0
        half = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan
        rec[f"{met}_mean"] = round(mean, 4); rec[f"{met}_std"] = round(sd, 4)
        rec[f"{met}_ci95_half"] = round(half, 4) if half == half else np.nan
        rec[f"{met}_ci95_lo"] = round(mean - half, 4) if half == half else np.nan
        rec[f"{met}_ci95_hi"] = round(mean + half, 4) if half == half else np.nan
    rows.append(rec)
agg = pd.DataFrame(rows)
agg["__o"] = agg["method"].map({m: i for i, m in enumerate(METHOD_ORDER)}).fillna(99)
agg = agg.sort_values(["dataset", "__o", "classifier"]).drop(columns="__o")
agg.to_csv(os.path.join(TABLES_DIR, "05_aggregated_mean_std_ci.csv"), index=False)

# publication AUC tables: rows=dataset, cols=method, 'mean +/- CI'
for clf in CLASSIFIERS:
    s = agg[agg["classifier"] == clf]
    pm = s.pivot(index="dataset", columns="method", values="AUC_mean")
    ph = s.pivot(index="dataset", columns="method", values="AUC_ci95_half")
    cols = [m for m in METHOD_ORDER if m in pm.columns]
    (pm[cols].round(4).astype(str) + " +/- " + ph[cols].round(4).astype(str)).to_csv(
        os.path.join(TABLES_DIR, f"table_auc_mean_ci_{clf}.csv"))
print("[R8-4] aggregated tables written.")

# mean-AUC frame (one value per dataset x method x classifier) for the stats
mf = agg[["dataset", "method", "classifier", "AUC_mean"]].rename(columns={"AUC_mean": "AUC"})

def nemenyi_cd(k, n):
    q = {2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031,9:3.102,10:3.164}.get(k, 3.164)
    return q * np.sqrt(k * (k + 1) / (6.0 * n))

# ---- R8-1: per-classifier Friedman + Nemenyi (primary 7-method set) ----
present = set(mf["method"].unique())
METHODS7 = [m for m in ["SMOTE","ADASYN","G-SMOTE","KMeans_CTGAN","CTGAN_MOS","ctdGAN","Q-G-CTGAN"] if m in present]
datasets = sorted(mf["dataset"].unique())
summary = []
for clf in CLASSIFIERS:
    blocks = []
    for ds in datasets:
        sub = mf[(mf.classifier == clf) & (mf.dataset == ds)][lambda d: d.method.isin(METHODS7)]
        sub = sub.set_index("method").reindex(METHODS7)
        if sub["AUC"].isna().any(): continue
        blocks.append(sub["AUC"].values.astype(float))
    bm = np.array(blocks); n, k = bm.shape
    stat, p = stats.friedmanchisquare(*[bm[:, i] for i in range(k)])
    nem = sp.posthoc_nemenyi_friedman(pd.DataFrame(bm, columns=METHODS7).values)
    nem.columns = METHODS7; nem.index = METHODS7
    nem.round(4).to_csv(os.path.join(TABLES_DIR, f"nemenyi_{clf}.csv"))
    ranks = np.vstack([pd.Series(r, index=METHODS7).rank(ascending=False).values for r in bm])
    avg = dict(zip(METHODS7, ranks.mean(0)))
    nsig = int((np.array(nem.values) < 0.05).sum() // 2)
    summary.append({"classifier": clf, "n_datasets": n, "n_methods": k,
                    "friedman_chi2": round(stat, 4), "friedman_p": round(p, 6),
                    "CD_0.05": round(nemenyi_cd(k, n), 3),
                    "best_method": LABELS[min(avg, key=avg.get)],
                    "QGCTGAN_avg_rank": round(avg.get("Q-G-CTGAN", np.nan), 2), "n_sig_pairs": nsig})
    pd.DataFrame([{ "method": LABELS[m], "avg_rank": round(avg[m], 3)} for m in sorted(avg, key=avg.get)]
                 ).to_csv(os.path.join(TABLES_DIR, f"avg_ranks_{clf}.csv"), index=False)
pd.DataFrame(summary).to_csv(os.path.join(TABLES_DIR, "friedman_summary.csv"), index=False)
print("[R8-1] per-classifier Friedman:"); print(pd.DataFrame(summary).to_string(index=False))

# ---- R8-2: Q-G-CTGAN (full) vs G-CTGAN (unfiltered) ----
rows2, detail = [], []
for clf in CLASSIFIERS:
    f = mf[(mf.classifier == clf) & (mf.method == "Q-G-CTGAN")].set_index("dataset")["AUC"]
    u = mf[(mf.classifier == clf) & (mf.method == "G-CTGAN")].set_index("dataset")["AUC"]
    common = [d for d in datasets if d in f.index and d in u.index]
    if not common: continue
    fv, uv = f.loc[common].values, u.loc[common].values; dl = fv - uv
    try: w = stats.wilcoxon(fv, uv)
    except ValueError: w = (np.nan, np.nan)
    t = stats.ttest_rel(fv, uv)
    rows2.append({"classifier": clf, "n_datasets": len(common),
                  "mean_AUC_full": round(fv.mean(), 4), "mean_AUC_unfiltered": round(uv.mean(), 4),
                  "mean_delta": round(dl.mean(), 4), "wins_full": int((dl > 0).sum()),
                  "ties": int((dl == 0).sum()), "losses_full": int((dl < 0).sum()),
                  "wilcoxon_p": round(float(w[1]), 4) if w[1] == w[1] else np.nan,
                  "paired_t_p": round(float(t.pvalue), 4)})
    for d in common:
        detail.append({"classifier": clf, "dataset": d,
                       "AUC_full": float(f.loc[d]), "AUC_unfiltered": float(u.loc[d]),
                       "delta": round(float(f.loc[d]) - float(u.loc[d]), 4)})
pd.DataFrame(rows2).to_csv(os.path.join(TABLES_DIR, "r8_2_full_vs_unfiltered.csv"), index=False)
pd.DataFrame(detail).to_csv(os.path.join(TABLES_DIR, "r8_2_per_dataset_detail.csv"), index=False)
print("[R8-2] full vs unfiltered:"); print(pd.DataFrame(rows2).to_string(index=False))
print("DONE. Tables in", TABLES_DIR)


merged 2298 rows from ['raw_classic.csv', 'raw_ctabgan.csv', 'raw_ctdgan.csv', 'raw_qgctgan.csv', 'raw_sdv.csv']
[R8-4] aggregated tables written.
[R8-1] per-classifier Friedman:
classifier  n_datasets  n_methods  friedman_chi2  friedman_p  CD_0.05 best_method  QGCTGAN_avg_rank  n_sig_pairs
        RF          16          7        26.7444    0.000162    2.252       SMOTE              3.62            4
      LGBM          16          7         9.3251    0.156101    2.252       SMOTE              3.50            0
       MLP          16          7        26.0447    0.000218    2.252     G-SMOTE              3.38            3
[R8-2] full vs unfiltered:
classifier  n_datasets  mean_AUC_full  mean_AUC_unfiltered  mean_delta  wins_full  ties  losses_full  wilcoxon_p  paired_t_p
        RF          16         0.8880               0.8851      0.0029         12     1            3      0.0115      0.0303
      LGBM          16         0.8871               0.8884     -0.0013          7     1     